# Bagré | 03: Actual evapotranspiration
    **Goal:** compare gridded ET estimates from MOD16A2GF and WaPOR v3. ET is the water transferred to the atmosphere through evaporation and transpiration. One mm of water depth equals one litre per square metre. Both datasets are modeled products, not independent ground measurements. WaPOR includes interception.
    MOD16 reports accumulated kg/m² per composite (numerically equivalent to mm), scaled by 0.1. WaPOR reports the mean daily mm/day in each dekad, scaled by 0.1. We convert both to monthly totals using actual overlap days. MOD16's overlap allocation assumes constant flux within its composite. No annual anomaly claims from four months.

## 1. Connect Drive and Earth Engine
    Use a CPU runtime. Run this cell after every runtime restart. You need a registered Earth Engine Google Cloud project (not your Gmail address). Set its project ID below. The Drive folder must be in your My Drive, or a shortcut with the same name.

In [ ]:
%pip -q install earthengine-api rasterio pyproj
import ee
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Bagre_ET_Water_Stress')
assert ROOT.is_dir(), f'Open Drive and confirm this folder exists: {ROOT}'
for sub in ['data', 'outputs', 'figures', 'qgis']:
    (ROOT / sub).mkdir(exist_ok=True)
PROJECT = 'YOUR_EARTH_ENGINE_PROJECT_ID'  # CHANGE THIS
assert PROJECT != 'YOUR_EARTH_ENGINE_PROJECT_ID', 'Set your registered Cloud project ID.'
try:
    ee.Initialize(project=PROJECT)
except ee.EEException:
    ee.Authenticate()
    ee.Initialize(project=PROJECT)
print('Earth Engine connection:', ee.Number(1).getInfo())

## 2. Study design
    Reuse the approximate Bagré pilot window from the earlier water project. It includes agricultural land and other land covers; it is not an official irrigation boundary. January–April 2021 is a single dry-season demonstration. WorldCover 2021 cropland is a proxy, not a field survey. All end dates are exclusive. Do not change dates or AOI halfway through the series.

In [ ]:
import calendar
from datetime import date, timedelta
AOI_BOUNDS = [-0.59, 11.41, -0.47, 11.52]  # west, south, east, north
AOI = ee.Geometry.Rectangle(AOI_BOUNDS, proj='EPSG:4326', geodesic=False)
CENTER = AOI.centroid(1)
CRS = 'EPSG:32630'
MONTHS = [(f'2021-{m:02d}-01', f'2021-{m+1:02d}-01') for m in range(1, 5)]
S2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(AOI)
L8 = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterBounds(AOI)
      .filter(ee.Filter.eq('PROCESSING_LEVEL', 'L2SP')))
MOD = ee.ImageCollection('MODIS/061/MOD16A2GF').filterBounds(AOI)
WAP = ee.ImageCollection('FAO/WAPOR/3/L1_AETI_D').filterBounds(AOI)
ERA = ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR').filterBounds(AOI)
WC = ee.ImageCollection('ESA/WorldCover/v200').first().select('Map')

### Download helper
    We download only the small pilot as GeoTIFF, with explicit CRS and scale. Earth Engine performs the processing remotely. Native thermal resolution remains about 100 m; a finer output grid would not create new information. The code checks that downloaded files have the expected number of bands and a valid coordinate system. Temporary files prevent incomplete downloads from masquerading as results.

In [ ]:
import requests
import rasterio
def download(image, path, scale, bands):
    path = Path(path)
    tmp = path.with_suffix('.partial.tif')
    url = image.select(bands).toFloat().clip(AOI).unmask(-9999, False).getDownloadURL({
        'name': path.stem, 'region': AOI, 'scale': scale,
        'crs': CRS, 'format': 'GEO_TIFF', 'filePerBand': False
    })
    with requests.get(url, stream=True, timeout=180) as response:
        if not response.ok:
            print(f'HTTP status: {response.status_code}')
            print('Earth Engine error:', response.text)
        response.raise_for_status()
        with tmp.open('wb') as f:
            for chunk in response.iter_content(1024 * 1024):
                f.write(chunk)
    with rasterio.open(tmp, 'r+') as src:
        assert src.count == len(bands) and src.crs.to_epsg() == 32630
        src.nodata = -9999
        for i, band in enumerate(bands, 1):
            src.set_band_description(i, band)
        print(path.name, src.shape, list(src.descriptions))
    tmp.replace(path)
    return path

## 3. Build monthly totals with explicit time weights
    Every period contributes a daily rate times the number of days falling in the target month. A fully covered month is required for each pixel. We reject invalid/fill values, preserve product gap-filling, and export coverage days; we do not pretend the gap-filled values are observations.

In [ ]:
def overlap_days(a, b, c, d):
    return max(0, (min(b, d) - max(a, c)).days)
assert overlap_days(date(2021,1,25), date(2021,2,2), date(2021,2,1), date(2021,3,1)) == 1
def interval_end(d, product):
    if product == 'mod16':
        return min(d + timedelta(days=8), date(d.year+1,1,1))
    if d.day == 1: return d.replace(day=11)
    if d.day == 11: return d.replace(day=21)
    assert d.day == 21, f'Unexpected WaPOR timestamp: {d}'
    return (d.replace(day=calendar.monthrange(d.year,d.month)[1]) + timedelta(days=1))
def month_et(start, end, product):
    a, b = date.fromisoformat(start), date.fromisoformat(end)
    collection = MOD if product == 'mod16' else WAP
    col = collection.filterDate(str(a-timedelta(days=10)), str(b)).sort('system:time_start')
    times = col.aggregate_array('system:time_start').getInfo()
    assert len(times) == len(set(times)), 'Duplicate dates; review source collection.'
    images = col.toList(len(times))
    et_parts, day_parts = [], []
    expected = (b-a).days
    total_weight = 0
    for i, stamp in enumerate(times):
        d = pd.to_datetime(stamp, unit='ms', utc=True).date()
        e = interval_end(d, product)
        weight = overlap_days(d, e, a, b)
        if not weight: continue
        total_weight += weight
        im = ee.Image(images.get(i))
        raw = im.select('ET' if product == 'mod16' else 'L1-AETI-D')
        # Exclude negative fills and implausible raw values. A generous 25 mm/day cap is a QC guard.
        rate = raw.multiply(0.1)
        if product == 'mod16': rate = rate.divide((e-d).days)
        valid = raw.gte(0).And(rate.lte(25))
        rate = rate.updateMask(valid)
        et_parts.append(rate.multiply(weight).rename('et_mm').toFloat())
        day_parts.append(rate.multiply(0).add(weight).rename('coverage_days').toFloat())
    assert total_weight == expected, (product, start, total_weight, expected)
    total = ee.ImageCollection.fromImages(et_parts).sum()
    days = ee.ImageCollection.fromImages(day_parts).sum()
    return total.updateMask(days.eq(expected)).addBands(days)

In [ ]:
for start, end in MONTHS:
    for product, scale in [('mod16', 500), ('wapor', 326)]:
        print(f'Processing {product}: {start[:7]}', flush=True)
        img = month_et(start, end, product)
        download(img, ROOT/'data'/f'{product}_{start[:7]}.tif', scale,
                 ['et_mm','coverage_days'])

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(15, 7), constrained_layout=True)
for j, (start, end) in enumerate(MONTHS):
    for i, product in enumerate(['mod16','wapor']):
        with rasterio.open(ROOT/'data'/f'{product}_{start[:7]}.tif') as src:
            a = src.read(1, masked=True)
        p = axes[i,j].imshow(a, cmap='YlGnBu', vmin=0, vmax=200)
        axes[i,j].set_title(f'{product} {start[:7]}')
        axes[i,j].axis('off')
        fig.colorbar(p, ax=axes[i,j], shrink=.7, label='mm/month')
        print(product, start[:7], 'valid fraction:', float(a.count()/a.size))
fig.savefig(ROOT/'figures'/'monthly_et_products.png', dpi=160)
plt.show()

**Checkpoint:** compare spatial patterns and missingness. Monthly totals must be nonnegative. Similarity is agreement, not proof of accuracy. Sources: [MOD16A2GF](https://developers.google.com/earth-engine/datasets/catalog/MODIS_061_MOD16A2GF), [WaPOR v3](https://developers.google.com/earth-engine/datasets/catalog/FAO_WAPOR_3_L1_AETI_D).